# Keeping .Stat Suite in sync with FMR

The FMR is the source of truth and .Stat Suite is expected to mirror it.
Publishing to .Stat only adds structures: a structure that is retired in
the FMR remains on .Stat until it is removed explicitly.

This course publishes real ECB exchange-rate structures to the FMR and
from the FMR to .Stat, creates a difference between the two, and then
removes from .Stat the structures that the FMR does not hold, so that
both match again.

| # | Step |
|---|------|
| 1 | Setup |
| 2 | Authentication |
| 3 | Prepare the ECB structures |
| 4 | Publish to the FMR |
| 5 | Publish FMR → .Stat |
| 6 | Create a difference |
| 7 | Compare FMR and .Stat |
| 8 | Clean up .Stat and verify |
| 9 | Recap and tear-down |

> **Requirements:** the pysdmx wheel and `helpers.py` in the folder of
> this notebook, and the SDMX Lab instance of the participant, with its
> user and its password (see *Configuration*).

> **Practice:** `fmr_stat_sync_practice.ipynb` is the exercise on the
> same steps.

> **Note:** the pysdmx interfaces used here are experimental and may
> change.

## 1 · Setup

pysdmx is installed and the required classes are imported. Three classes
do the work:

- `RegistryMaintenanceClient` writes to the FMR,
- `RestService` reads from the FMR and from .Stat,
- `StatUploader` writes to and deletes from .Stat.

All the exchanges use **SDMX-ML 2.1**, except the upload to the FMR,
which pysdmx sends in SDMX-JSON.

In [ ]:
%pip install --quiet ./pysdmx-1.24.0rc1-py3-none-any.whl[all]
%pip install --quiet certifi

### Configuration

The settings are read from environment variables by `helpers.py`, in the
folder of this notebook, when it is imported. Each participant has an
SDMX Lab instance, with its user and its password: the next cell sets
`PREFIX` and requests them, unless they are already set in the
environment; the password is not displayed. The kernel must be restarted
to change a setting after the imports.

| Variable | Meaning | Default |
|---|---|---|
| `SDMX_LAB` | Identifier of the SDMX Lab instance, the first part of its address (`<id>.sdmx.solutions`) | requested |
| `SDMX_LAB_USER` and `SDMX_LAB_PASSWORD` | User and password of the instance | requested |
| `PREFIX` | Prefix of the ids of the structures | `PX_SYNC` |
| `AGENCY` | Agency of the structures | `BIS` |
| `SPACE` | Dataspace of .Stat | `demo-design` |

The addresses of the FMR, of the .Stat services and of Keycloak are
derived from the instance.

In [ ]:
import os
from getpass import getpass

# Every structure of the tutorial starts with this prefix, which tells them
# apart from the other structures of the agency. Only letters, digits and
# underscores are allowed.
os.environ["PREFIX"] = "PX_SYNC"

# The SDMX Lab instance of the participant, its user and its password are
# requested, unless they are already set in the environment. The password
# is not displayed.
if not os.environ.get("SDMX_LAB"):
    os.environ["SDMX_LAB"] = input("SDMX Lab instance (<id>.sdmx.solutions): ")
if not os.environ.get("SDMX_LAB_USER"):
    os.environ["SDMX_LAB_USER"] = input("User: ")
    os.environ["SDMX_LAB_PASSWORD"] = getpass("Password: ")

In [ ]:
from io import BytesIO

import pandas as pd
from helpers import (
    AGENCY,
    DELETION_ORDER,
    ECB_STRUCTURES,
    PREFIX,
    STAT_FORMAT,
    build_structures,
    check_connections,
    fmr_client,
    fmr_service,
    list_urns,
    stat_service,
    stat_uploader,
)

from pysdmx.api.fmr.maintenance import StructureAction
from pysdmx.api.qb import (
    StructureDetail,
    StructureQuery,
    StructureReference,
    StructureType,
)
from pysdmx.errors import Invalid, NotFound
from pysdmx.io import read_sdmx
from pysdmx.model import AgencyScheme

## 2 · Authentication

The user and the password of the SDMX Lab instance serve both services.

**FMR.** The FMR takes HTTP Basic authentication.
`RegistryMaintenanceClient`, which writes, takes the user and the
password as arguments. `RestService`, which reads, only takes a bearer
token, so `fmr_service()` sets the HTTP Basic header itself.

```python
fmr = RegistryMaintenanceClient(
    FMR_ENDPOINT, user=LAB_USER, password=LAB_PASSWORD
)
```

**.Stat Suite.** The NSI web service and the Transfer service take a
bearer token issued by the Keycloak of the instance. pysdmx requests it
with the user and the password. The token is valid for a few minutes,
and `get_token()` renews it when it expires:

```python
auth = KeycloakAuthentication(
    KEYCLOAK, user=LAB_USER, password=LAB_PASSWORD, client_id="stat-suite"
)
stat = StatUploader(
    nsi_endpoint=NSI,
    transfer_endpoint=TRANSFER,
    dataspace=SPACE,
    token=auth.get_token(),
)
```

`helpers.py` wraps these calls: `fmr_client()` and `stat_uploader()`
return the clients with write access, and `fmr_service()` and
`stat_service()` the readers. `check_connections()` tests them with
read-only queries: both services must be connected before going on.

In [ ]:
fmr = fmr_client()
check_connections()

## 3 · Prepare the ECB structures

The ECB exchange-rate structures are read from the web service of the
ECB ([EXR](https://data.ecb.europa.eu/data/datasets/EXR)).

In [ ]:
ecb = read_sdmx(ECB_STRUCTURES, validate=False)
pd.Series([type(s).__name__ for s in ecb.structures]).value_counts()

### Move them to the agency and build two dataflows

Structures can only be written under the agency of the participant.
`build_structures` (in `helpers.py`) copies the ECB concept scheme and
codelists to that agency and adds the prefix to their ids; their content
does not change. It then builds two dataflows on top of them:

- `EXR`, which **stays** in the FMR (the list `keep`);
- `EXR_OLD`, an older version with a `UNIT` attribute (`CL_UNIT`),
  which is **retired** (the list `legacy`).

> **Note:** `SDMX:AGENCIES` is left out: the agency scheme is not
> maintained by this process.

In [ ]:
keep, legacy = build_structures(ecb)
pd.DataFrame(
    [("keep", s.short_urn, s.name) for s in keep]
    + [("legacy", s.short_urn, s.name) for s in legacy],
    columns=["list", "short URN", "name"],
)

## 4 · Publish to the FMR

### Upload to the FMR

`StructureAction.Replace` overwrites the existing structures, so the
cell can be run again. `legacy` is not uploaded: it stands for
structures that have been retired in the FMR.

In [ ]:
fmr.put_structures(keep, action=StructureAction.Replace)
print("uploaded to FMR:", [s.short_urn for s in keep])

### Download from the FMR

The dataflow is requested with `references=all`, so that the FMR returns
it together with its DSD, its concept scheme and its codelists.

In [ ]:
query = StructureQuery(
    artefact_type=StructureType.DATAFLOW,
    agency_id=AGENCY,
    resource_id=f"{PREFIX}_DF_EXR",
    version="1.0",
    detail=StructureDetail.FULL,
    references=StructureReference.ALL,
)
graph = read_sdmx(BytesIO(fmr_service().structure(query)), validate=False)
[s.short_urn for s in graph.structures]

## 5 · Publish FMR → .Stat

### Start from a clean .Stat

.Stat rejects a structure that already exists, so the structures left by
a previous run are removed first. Each list ends with its DSD and its
dataflow, so reversing it deletes the dependents first.

In [ ]:
stat = stat_uploader()
for artefact in [*reversed(legacy), *reversed(keep)]:
    try:
        stat.delete_structure(artefact)
        print("removed  ", artefact.short_urn)
    except (Invalid, NotFound):  # noqa: PERF203
        print("not there", artefact.short_urn)

### Submit to .Stat

`SDMX:AGENCIES` is dropped, because the agency scheme is not maintained
by this process, and the rest is submitted. A structure rejected by .Stat is
reported in the messages of the result.

In [ ]:
to_stat = [s for s in graph.structures if not isinstance(s, AgencyScheme)]
result = stat_uploader().submit_structure(to_stat, STAT_FORMAT)
print("success:", result.success)
for message in result.messages:
    print(" -", message)

## 6 · Create a difference

`EXR_OLD` is submitted to .Stat only. This reproduces a structure that
has been retired in the FMR but remains on .Stat.

In [ ]:
result = stat_uploader().submit_structure(legacy, STAT_FORMAT)
print("success:", result.success, [s.short_urn for s in legacy])

## 7 · Compare FMR and .Stat

`list_urns` (in `helpers.py`) lists the structures of the agency on a
service, one type at a time (.Stat rejects the queries for all types)
and as stubs (ids and versions only), and returns their short URNs.

> **Note:** only the ids that start with the prefix are listed, because
> the agency may hold structures that are not part of the tutorial.

The FMR is the reference, so its list must be complete. Without valid
credentials the list of the FMR is empty, and every structure on .Stat
would then be taken as retired; the cell stops in that case.

In [ ]:
fmr_urns = list_urns(fmr_service())
if not fmr_urns:
    raise RuntimeError("The FMR list is empty: check the user and password.")
stat_urns = list_urns(stat_service())

pd.DataFrame({"short URN": sorted(fmr_urns | stat_urns)}).assign(
    in_FMR=lambda d: d["short URN"].isin(fmr_urns),
    in_Stat=lambda d: d["short URN"].isin(stat_urns),
)

### What .Stat has that the FMR does not

The difference is the set of structures to remove. The codelists shared
with `EXR` (`CL_CURRENCY`, …) are still in the FMR, so they are not part
of it.

In [ ]:
extras = stat_urns - fmr_urns  # present in .Stat, absent in FMR
sorted(extras)

> **Warning:** the list must be checked before anything is deleted. If
> the list of the FMR were incomplete, valid structures would appear in
> it as well.

## 8 · Clean up .Stat and verify

### Deletion order

Structures depend on each other:

```
Dataflow ──▶ DataStructure ──▶ ConceptScheme / Codelist
   ▲
DataConstraint (CR_A_<dataflow>, created by data imports)
```

.Stat rejects the deletion of a structure that is still in use (409), as
shown by an attempt to delete the DSD first:

In [ ]:
wrong_first = sorted(u for u in extras if u.startswith("DataStructure="))
try:
    stat_uploader().delete_structure(wrong_first)
except Invalid as err:
    print("Rejected, as expected:\n", err)

The list is therefore sorted so that the dependents are deleted first:
constraints, dataflows, DSDs, and then concept schemes and codelists
(`DELETION_ORDER`, in `helpers.py`).

In [ ]:
ordered = sorted(extras, key=lambda u: DELETION_ORDER[u.split("=")[0]])
ordered

### Delete

`delete_structure` deletes the list in order and stops at the first
error. In that case, the cause must be fixed and the cell run again with
the remaining structures.

In [ ]:
try:
    results = stat_uploader().delete_structure(ordered)
    for urn, result in zip(ordered, results):
        print("deleted", urn, "|", "; ".join(result.messages))
except (Invalid, NotFound) as err:
    print("FAILED:", err)

### Verify

.Stat is listed again: no extra structure should remain, and every
structure of the FMR should still be there.

In [ ]:
stat_after = list_urns(stat_service())
print("still extra on .Stat:", sorted(stat_after - fmr_urns) or "none")
print("FMR structures kept: ", sorted(stat_after & fmr_urns))

## 9 · Recap

| Step | pysdmx |
|---|---|
| Authenticate to the FMR | `RegistryMaintenanceClient` (user and password) |
| Authenticate to .Stat | `KeycloakAuthentication` + `StatUploader` |
| Read structures | `read_sdmx` |
| Publish to FMR | `RegistryMaintenanceClient.put_structures` |
| Read FMR / .Stat | `RestService` + `StructureQuery` |
| Publish to .Stat | `StatUploader.submit_structure` |
| Compare | set difference of short URNs |
| Clean up .Stat | `StatUploader.delete_structure` |

- The FMR is the reference: its list must be complete.
- The difference must be checked before anything is deleted.
- Dependents are deleted first (including `CR_A_<dataflow>` after data
  imports).
- The deletion stops at the first error.

### Tear-down

`keep` is removed from .Stat, so that the notebook can be run again. The
copies in the FMR remain, because pysdmx cannot delete from the FMR.

In [ ]:
try:
    stat_uploader().delete_structure(list(reversed(keep)))
    print("removed from .Stat:", [s.short_urn for s in reversed(keep)])
except (Invalid, NotFound) as err:
    print("FAILED:", err)

### Practice

`fmr_stat_sync_practice.ipynb` is the exercise on the same steps: five
of them are to be completed by the participant. The answer is in
`fmr_stat_sync_practice.py`.